In [1]:
import faiss
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoProcessor

/Users/paullin/Documents/SPROUT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 22:53:48.026000 59095 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
name = "google/siglip2-base-patch16-224"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(name).to(device).eval()
processor = AutoProcessor.from_pretrained(name)

def embed(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.inference_mode():
        output = model.vision_model(**inputs)
        embedding = output.pooler_output
        embedding = F.normalize(embedding, p=2, dim=-1)

    return embedding.cpu().numpy().astype("float32")


Loading weights: 100%|██████████| 408/408 [00:00<00:00, 14683.27it/s]


In [ ]:
image_paths = [
    "image.png",
    "image2.png",
    "image3.png",
]

embeddings = [embed(path) for path in image_paths]

: 

In [ ]:
# Build HNSW approximate nearest-neighbor index
dimension = embeddings.shape[1]

index = faiss.IndexHNSWFlat(
    dimension,
    32,
    faiss.METRIC_INNER_PRODUCT,
)

index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64

index.add(embeddings)

print(f"Indexed {index.ntotal} images")